In [12]:
import sys
print(sys.executable)
!which python

/Users/carolinanami/Desktop/Ironhack/Week 1/.conda/bin/python
/Users/carolinanami/Desktop/Ironhack/Week 1/.conda/bin/python


In [13]:
import sys
!{sys.executable} -m pip install cohere

zsh:1: no such file or directory: /Users/carolinanami/Desktop/Ironhack/Week


In [14]:
import sys
python_path = sys.executable
!"{python_path}" -m pip install cohere

  Using cached cohere-5.20.6-py3-none-any.whl.metadata (3.6 kB)
  Using cached types_requests-2.32.4.20260107-py3-none-any.whl.metadata (2.0 kB)
Using cached cohere-5.20.6-py3-none-any.whl (323 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.1 MB/s  0:00:00
Using cached types_requests-2.32.4.20260107-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cohere]2m2/3 [cohere]


In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Load environment variables from .env file
load_dotenv()

# Verify API keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")
if not os.getenv("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY not found in environment variables. Please create a .env file with your API key.")

print("✅ API keys loaded successfully!")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ LLM and Embeddings initialized!")

/Users/carolinanami/Desktop/Ironhack/Week 1/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ API keys loaded successfully!
✅ LLM and Embeddings initialized!


In [2]:
# Query rewriting example
def rewrite_query(original_query):
    prompt = f"Rewrite this query to be more specific and better suited for document retrieval: {original_query}"
    response = llm.invoke(prompt)
    return response.content

# Test it out
original = "Tell me about AI"
rewritten = rewrite_query(original)
print(f"Original: {original}")
print(f"Rewritten: {rewritten}")

Original: Tell me about AI
Rewritten: Please provide an overview of artificial intelligence, including its key concepts, applications, and recent advancements in the field. Additionally, include information on ethical considerations and challenges associated with AI development.


In [3]:
# Sub-query decomposition - breaking complex questions into smaller ones
def decompose_query(complex_query):
    prompt = f"Break this complex query into 2-3 simpler sub-queries: {complex_query}"
    response = llm.invoke(prompt)
    return response.content

# Test it out
complex = "How does machine learning compare to deep learning and what are their applications?"
sub_queries = decompose_query(complex)
print(f"Complex query: {complex}")
print(f"\nSub-queries:\n{sub_queries}")

Complex query: How does machine learning compare to deep learning and what are their applications?

Sub-queries:
To break down the complex query into simpler sub-queries, we can focus on distinct aspects of the comparison and applications of machine learning and deep learning. Here are three sub-queries:

1. **What are the key differences between machine learning and deep learning?**
   
2. **What are the main applications of machine learning?**

3. **What are the main applications of deep learning?**

These sub-queries allow for a clearer exploration of the concepts and their respective uses.


In [4]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a parent document (like a long paragraph)
parent_docs = [
    Document(page_content="Machine learning is a subset of artificial intelligence. Deep learning uses neural networks with many layers. Natural language processing helps computers understand human language. Computer vision enables machines to see and identify objects. Reinforcement learning trains models through trial and error.")
]

print(f"Parent document length: {len(parent_docs[0].page_content)} characters")
print(f"Parent document: {parent_docs[0].page_content}")

Parent document length: 304 characters
Parent document: Machine learning is a subset of artificial intelligence. Deep learning uses neural networks with many layers. Natural language processing helps computers understand human language. Computer vision enables machines to see and identify objects. Reinforcement learning trains models through trial and error.


In [5]:
# Split into small chunks (like cutting a long paragraph into smaller sentences)
small_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10)
small_chunks = small_splitter.split_documents(parent_docs)

# Add parent ID to each chunk (like labeling each piece with which article it came from)
for i, chunk in enumerate(small_chunks):
    chunk.metadata["parent_id"] = 0  # Reference to parent document

print(f"Created {len(small_chunks)} small chunks from 1 parent document")
print("\nAll chunks:")
for i, chunk in enumerate(small_chunks):
    print(f"Chunk {i+1}: {chunk.page_content}")

Created 8 small chunks from 1 parent document

All chunks:
Chunk 1: Machine learning is a subset of artificial
Chunk 2: intelligence. Deep learning uses neural networks
Chunk 3: networks with many layers. Natural language
Chunk 4: language processing helps computers understand
Chunk 5: human language. Computer vision enables machines
Chunk 6: machines to see and identify objects.
Chunk 7: objects. Reinforcement learning trains models
Chunk 8: models through trial and error.


In [6]:
# LLM-based reranking
def rerank_documents(query, documents, top_k=2):
    # Score each document (like giving each answer a grade from 0 to 1)
    scores = []
    for doc in documents:
        prompt = f"Rate the relevance of this document to the query '{query}' on a scale of 0-1. Return only the number: {doc.page_content}"
        response = llm.invoke(prompt)
        # Extract score
        try:
            score = float(response.content.strip())
            scores.append((score, doc))
            print(f"Score for '{doc.page_content[:30]}...': {score}")
        except:
            scores.append((0.5, doc))
            print(f"Couldn't parse score, using default 0.5 for '{doc.page_content[:30]}...'")
    
    # Sort by score (highest to lowest) and return top_k
    scores.sort(reverse=True, key=lambda x: x[0])
    print(f"\nTop {top_k} results after reranking:")
    return [doc for score, doc in scores[:top_k]]

# Create sample documents to test
sample_docs = [
    Document(page_content="Machine learning is a subset of artificial intelligence that uses data to learn."),
    Document(page_content="The weather today is sunny and warm with a slight chance of rain."),
    Document(page_content="Deep learning uses neural networks with many layers for pattern recognition."),
    Document(page_content="Artificial intelligence aims to create machines that can think and learn like humans.")
]

# Test it with a query about machine learning
query = "What is machine learning?"
print(f"Query: {query}\n")
print("Scoring each document...\n")

reranked = rerank_documents(query, sample_docs)

print("\nFinal ranked results:")
for i, doc in enumerate(reranked):
    print(f"{i+1}. {doc.page_content}")

Query: What is machine learning?

Scoring each document...

Score for 'Machine learning is a subset o...': 1.0
Score for 'The weather today is sunny and...': 0.0
Score for 'Deep learning uses neural netw...': 0.5
Score for 'Artificial intelligence aims t...': 0.5

Top 2 results after reranking:

Final ranked results:
1. Machine learning is a subset of artificial intelligence that uses data to learn.
2. Deep learning uses neural networks with many layers for pattern recognition.


In [7]:
# Add metadata to documents
from datetime import datetime

documents_with_metadata = [
    Document(
        page_content="New regulation on AI data privacy under EU AI Act",
        metadata={"date": "2024-01-15", "category": "regulations", "source": "legal", "risk_level": "high"}
    ),
    Document(
        page_content="Old data protection regulation from 2020",
        metadata={"date": "2020-03-10", "category": "regulations", "source": "legal", "risk_level": "low"}
    ),
    Document(
        page_content="Technical documentation for AI model deployment",
        metadata={"date": "2024-02-20", "category": "technical", "source": "docs", "risk_level": "medium"}
    ),
    Document(
        page_content="Guidelines for high-risk AI systems under EU AI Act",
        metadata={"date": "2024-01-20", "category": "regulations", "source": "legal", "risk_level": "high"}
    )
]

# Filter by metadata - like sorting your files by date and category
def filter_by_metadata(docs, date_cutoff=None, category=None, risk_level=None):
    filtered = docs
    if date_cutoff:
        filtered = [d for d in filtered if d.metadata.get("date", "") >= date_cutoff]
        print(f"📅 Filtered by date: after {date_cutoff}")
    if category:
        filtered = [d for d in filtered if d.metadata.get("category") == category]
        print(f"📂 Filtered by category: {category}")
    if risk_level:
        filtered = [d for d in filtered if d.metadata.get("risk_level") == risk_level]
        print(f"⚠️ Filtered by risk level: {risk_level}")
    return filtered

# Example 1: Get recent regulations only
print("🔍 Example 1: Recent regulations about AI\n")
recent_regs = filter_by_metadata(documents_with_metadata, date_cutoff="2024-01-01", category="regulations")
print(f"\nFound {len(recent_regs)} recent regulations:")
for doc in recent_regs:
    print(f"- {doc.page_content} ({doc.metadata['date']}, risk: {doc.metadata['risk_level']})")

# Example 2: Get high-risk documents only
print("\n" + "="*50 + "\n")
print("🔍 Example 2: High-risk AI documents\n")
high_risk = filter_by_metadata(documents_with_metadata, risk_level="high")
print(f"\nFound {len(high_risk)} high-risk documents:")
for doc in high_risk:
    print(f"- {doc.page_content} (category: {doc.metadata['category']})")

🔍 Example 1: Recent regulations about AI

📅 Filtered by date: after 2024-01-01
📂 Filtered by category: regulations

Found 2 recent regulations:
- New regulation on AI data privacy under EU AI Act (2024-01-15, risk: high)
- Guidelines for high-risk AI systems under EU AI Act (2024-01-20, risk: high)


🔍 Example 2: High-risk AI documents

⚠️ Filtered by risk level: high

Found 2 high-risk documents:
- New regulation on AI data privacy under EU AI Act (category: regulations)
- Guidelines for high-risk AI systems under EU AI Act (category: regulations)


In [18]:
# Compare LLM-based reranking vs Cohere's dedicated reranker
import cohere

# Initialize Cohere client
co = cohere.Client(os.getenv("COHERE_API_KEY"))

# Sample documents about AI literacy (from your PDF)
advanced_docs = [
    Document(page_content="AI literacy refers to understanding AI capabilities and limitations, as well as the ability to effectively integrate AI tools into workflows."),
    Document(page_content="The AI Act requires providers and deployers to ensure sufficient AI literacy of their staff and other persons dealing with AI systems."),
    Document(page_content="Weather forecast models use machine learning to predict temperature and rainfall patterns."),
    Document(page_content="AI literacy programs should be tailored to different roles: leadership, technical teams, and frontline employees."),
    Document(page_content="The stock market analysis uses deep learning to predict price movements based on historical data."),
    Document(page_content="Article 4 of the AI Act emphasizes AI literacy considering technical knowledge, experience, and context of use.")
]

query = "What is AI literacy and what does the AI Act require?"

print(f"🔍 Query: {query}\n")
print("="*70 + "\n")

# Method 1: LLM-based reranking
print("📊 METHOD 1: LLM-BASED RERANKING")
print("-" * 40)

def llm_rerank(query, docs, top_k=3):
    scores = []
    for doc in docs:
        prompt = f"Rate the relevance of this document to the query '{query}' on a scale of 0-1. Return only the number: {doc.page_content}"
        response = llm.invoke(prompt)
        try:
            score = float(response.content.strip())
            scores.append((score, doc))
        except:
            scores.append((0.5, doc))
    
    scores.sort(reverse=True, key=lambda x: x[0])
    return [(score, doc) for score, doc in scores[:top_k]]

llm_results = llm_rerank(query, advanced_docs)
print("Top results:")
for i, (score, doc) in enumerate(llm_results):
    print(f"{i+1}. Score: {score} - {doc.page_content[:100]}...")

print("\n" + "="*70 + "\n")

# Method 2: Cohere dedicated reranker
print("🚀 METHOD 2: COHERE DEDICATED RERANKER")
print("-" * 40)

# Prepare documents for Cohere (just the text)
docs_text = [doc.page_content for doc in advanced_docs]

# Use Cohere's rerank
cohere_results = co.rerank(
    query=query,
    documents=docs_text,
    top_n=3,
    model="rerank-english-v3.0"
)

print("Top results:")
for i, result in enumerate(cohere_results.results):
    # Get the document using the index from the result
    doc_index = result.index
    doc_text = docs_text[doc_index]
    print(f"{i+1}. Relevance Score: {result.relevance_score:.3f} - {doc_text[:100]}...")

print("\n" + "="*70 + "\n")

# Compare the results
print("📈 COMPARISON: Which documents did each method pick?")
print("-" * 40)

llm_docs = [doc.page_content for score, doc in llm_results]
cohere_docs = [docs_text[result.index] for result in cohere_results.results]

print("LLM Reranker picked:")
for i, doc in enumerate(llm_docs):
    print(f"  {i+1}. {doc[:80]}...")

print("\nCohere Reranker picked:")
for i, doc in enumerate(cohere_docs):
    print(f"  {i+1}. {doc[:80]}...")

# Check for overlap
common = set(llm_docs).intersection(set(cohere_docs))
print(f"\n✅ Documents both methods agreed on: {len(common)} out of 3")

🔍 Query: What is AI literacy and what does the AI Act require?


📊 METHOD 1: LLM-BASED RERANKING
----------------------------------------
Top results:
1. Score: 1.0 - The AI Act requires providers and deployers to ensure sufficient AI literacy of their staff and othe...
2. Score: 1.0 - Article 4 of the AI Act emphasizes AI literacy considering technical knowledge, experience, and cont...
3. Score: 0.5 - AI literacy refers to understanding AI capabilities and limitations, as well as the ability to effec...


🚀 METHOD 2: COHERE DEDICATED RERANKER
----------------------------------------
Top results:
1. Relevance Score: 0.999 - The AI Act requires providers and deployers to ensure sufficient AI literacy of their staff and othe...
2. Relevance Score: 0.998 - AI literacy refers to understanding AI capabilities and limitations, as well as the ability to effec...
3. Relevance Score: 0.997 - Article 4 of the AI Act emphasizes AI literacy considering technical knowledge, experience, and cont...

In [1]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
import time

# Initialize Pinecone (make sure you have PINECONE_API_KEY in .env)
index_name = "ai-literacy-lab"

print("✅ Pinecone ready")

✅ Pinecone ready


/Users/carolinanami/Desktop/Ironhack/Week 1/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Install pypdf if not already installed
!pip install pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF - you'll need to put the file in the correct location
# First, let's check if the file exists where we expect
import os

# Create data folder if it doesn't exist
os.makedirs("data", exist_ok=True)

print("📁 Please copy your PDF file to: data/Living_Repository_AI_Literacy_Practices.pdf")
print("After you've copied the file, run the next cell.")

📁 Please copy your PDF file to: data/Living_Repository_AI_Literacy_Practices.pdf
After you've copied the file, run the next cell.


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Go up one level to reach the data folder
file_path = "../data/Lab: Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act….pdf"

# Load the PDF
loader = PyPDFLoader(file_path)
documents = loader.load()

print(f"📄 Loaded {len(documents)} pages from PDF")
print(f"First page preview: {documents[0].page_content[:200]}...\n")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Size of each chunk
    chunk_overlap=50,  # Overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"✂️ Split into {len(chunks)} chunks")
print(f"First chunk preview: {chunks[0].page_content[:150]}...")

ValueError: File path ../data/Lab: Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act….pdf is not a valid file or url

In [5]:
import os

# List all files in the data folder
print("Files in data folder:")
for file in os.listdir("data"):
    print(f"  - {file}")
    
# Check current working directory
print(f"\nCurrent working directory: {os.getcwd()}")

Files in data folder:
  - Lab: Relevance Scoring and Rerankers for Trustworthy AI & EU AI Act….pdf

Current working directory: /Users/carolinanami/Desktop/Ironhack/Week 3/Day 4/Week 3 Day 4 Lab/notebooks


In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use the correct filename
file_path = "../data/Living_Repository_AI_Literacy_Practices.pdf"

# Check if file exists first
import os
if os.path.exists(file_path):
    print(f"✅ Found file: {file_path}")
    
    # Load the PDF
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    
    print(f"📄 Loaded {len(documents)} pages from PDF")
    print(f"First page preview: {documents[0].page_content[:200]}...\n")
    
    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    chunks = text_splitter.split_documents(documents)
    print(f"✂️ Split into {len(chunks)} chunks")
    print(f"First chunk preview: {chunks[0].page_content[:150]}...")
    
else:
    print(f"❌ File not found: {file_path}")
    print("\nFiles in data folder:")
    for file in os.listdir("../data"):
        print(f"  - {file}")

✅ Found file: ../data/Living_Repository_AI_Literacy_Practices.pdf
📄 Loaded 73 pages from PDF
First page preview: 1 
 
Living Repository of  
AI Literacy Practices – v. 16.04.2025 
 
Living Repository of  
AI Literacy Practices  
v.16.04.2025 
 
 
Disclaimer 
The following document is a living repository of AI li...

✂️ Split into 533 chunks
First chunk preview: 1 
 
Living Repository of  
AI Literacy Practices – v. 16.04.2025 
 
Living Repository of  
AI Literacy Practices  
v.16.04.2025 
 
 
Disclaimer 
The ...


In [11]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

# Initialize embeddings (same as before)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Index name (same as you created)
index_name = "ai-literacy-lab"

print("⏳ Adding documents to Pinecone... This may take a minute or two.")

# Add documents to Pinecone
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

print("✅ Documents successfully added to Pinecone!")
print(f"   Added {len(chunks)} chunks to index '{index_name}'")

⏳ Adding documents to Pinecone... This may take a minute or two.
✅ Documents successfully added to Pinecone!
   Added 533 chunks to index 'ai-literacy-lab'


In [12]:
# Connect to the existing Pinecone index
vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embeddings
)

# Test query
query = "What is AI literacy according to the AI Act?"

print(f"🔍 Query: {query}\n")
print("="*60 + "\n")

# Simple similarity search
print("📊 BASIC SIMILARITY SEARCH (without reranking):")
basic_results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(basic_results):
    print(f"\n{i+1}. [Score: vector similarity]")
    print(f"   {doc.page_content[:200]}...")
    print(f"   Metadata: {doc.metadata}")

print("\n" + "="*60 + "\n")

# Now let's use our reranker on these results
print("🚀 WITH COHERE RERANKING:")
docs_text = [doc.page_content for doc in basic_results]

cohere_results = co.rerank(
    query=query,
    documents=docs_text,
    top_n=3,
    model="rerank-english-v3.0"
)

for i, result in enumerate(cohere_results.results):
    doc_index = result.index
    doc_text = docs_text[doc_index]
    print(f"\n{i+1}. Cohere Score: {result.relevance_score:.3f}")
    print(f"   {doc_text[:200]}...")

🔍 Query: What is AI literacy according to the AI Act?


📊 BASIC SIMILARITY SEARCH (without reranking):

1. [Score: vector similarity]
   Key AI literacy pillars include:  
1) AI Governance Model: implementing an AI Organisational Model and an AI Code of Conduct 
defining and documenting roles, responsibilities, principles, processes, r...
   Metadata: {'creationdate': '2025-05-24T20:32:15+02:00', 'creator': 'PyPDF', 'moddate': '2025-05-24T20:32:15+02:00', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_actionid': 'bf1a5e57-4e67-4a7b-9a2d-6232fdeec5e7', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_contentbits': '0', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_enabled': 'true', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_method': 'Standard', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_name': 'Commission Use', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_setdate': '2024-12-16T15:28:28Z', 'msip_label_6bd9ddd1-4d20-43f6-abfa-fc3c07406f94_siteid': 'b24c8b06-522c-46fe-

NameError: name 'co' is not defined

In [13]:
import cohere
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize Cohere client
co = cohere.Client(os.getenv("COHERE_API_KEY"))

print("✅ Cohere client re-initialized")

# Now let's rerun the reranking part
print("\n🚀 WITH COHERE RERANKING:")
docs_text = [doc.page_content for doc in basic_results]

cohere_results = co.rerank(
    query=query,
    documents=docs_text,
    top_n=3,
    model="rerank-english-v3.0"
)

for i, result in enumerate(cohere_results.results):
    doc_index = result.index
    doc_text = docs_text[doc_index]
    print(f"\n{i+1}. Cohere Score: {result.relevance_score:.3f}")
    print(f"   {doc_text[:200]}...")

✅ Cohere client re-initialized

🚀 WITH COHERE RERANKING:

1. Cohere Score: 0.996
   requirements under the EU AI Act. The primary challenge we have addressed is the limited operational 
guidance on AI literacy provided by Article 4 of the AI Act. Since the article doesn't provide spe...

2. Cohere Score: 0.974
   Key AI literacy pillars include:  
1) AI Governance Model: implementing an AI Organisational Model and an AI Code of Conduct 
defining and documenting roles, responsibilities, principles, processes, r...

3. Cohere Score: 0.745
   AI opportunities and reduce risk, through transparency, human oversight, and AI literacy. 
 
On the AI literacy approach 
Status: Partially rolled-out 
Target group: Organisation's staff , including r...
